# Dino Run: Visual Deep Reinforcement Learning

This notebook connects a convolutional Q-network to the original JavaScript T-Rex game. The agent receives four recent screenshots, chooses **run** or **jump**, and learns from the reward returned after the game advances. It uses Double DQN targets, prioritized experience replay, a slowly updated target network, and a behavior policy held fixed during each episode.

The notebook is organized as a sequence of modules: configure the local game and browser; define the network and replay memory; wrap the browser game as a step-by-step environment; train a new policy; load the supplied `dino_model.pt`; evaluate that loaded policy over independent episodes; and optionally show an episode as an inline GIF. The training results and the supplied checkpoint are separate: the checkpoint-loading cell creates `net` from `dino_model.pt`, so the following evaluation and animation use that checkpoint even if the earlier training cell ran.

**Before running:** keep `dinorl_in_memory.ipynb`, `dino_model.pt`, `tools/geckodriver.exe`, and the `t-rex-runner-gh-pages` folder in the project directory. Use the Conda `base` kernel with PyTorch, NumPy, Pillow, Selenium, and IPython installed, and have Firefox available. Run the cells from top to bottom. Training, five-episode evaluation, and animation each launch a browser and take additional time. The game and GIF are served or held in memory; the notebook does not save training checkpoints, CSV logs, or GIF files. Jupyter may still save the notebook itself, and Firefox may use its own browser cache.

The experiment uses a pixel observation and the game’s original physics. It does not feed the numeric game-state fields to the neural network. A short run is a code demonstration, not proof that the policy has converged or performs reliably.

In [ ]:
import base64, copy, io, json, math, mimetypes, random, shutil, threading, time, zipfile
from collections import deque
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import unquote, urlsplit

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display, HTML
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

DEVICE = torch.device('cpu')
torch.set_num_threads(1)
RUN_TRAINING = False  # Set True to run the 120-second training experiment.
CHECKPOINT = Path('dino_model.pt')  # Existing file; never overwritten.
GECKODRIVER = (str(Path('tools/geckodriver.exe').resolve())
               if Path('tools/geckodriver.exe').is_file() else shutil.which('geckodriver'))
# Or supply the absolute path to an installed geckodriver executable.
FIREFOX_BINARY = None  # Optional absolute Firefox executable path.
HEADLESS = True
ACTION_REPEAT = 4
GAMMA = 0.99
SEED = 20261015
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CPU mode. Run training, evaluation, and demo cells individually.')


# Initialize the embedded game in the same cell as configuration.
GAME_DIR = Path('t-rex-runner-gh-pages').resolve()
if not (GAME_DIR / 'index.html').is_file() or not (GAME_DIR / 'state-control.js').is_file():
    raise FileNotFoundError(f'Game folder or state-control.js not found: {GAME_DIR}')

class LocalGameHandler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=str(GAME_DIR), **kwargs)
    def end_headers(self):
        self.send_header('Cache-Control', 'no-store')
        super().end_headers()
    def log_message(self, *args):
        pass

if globals().get('asset_server') is not None:
    asset_server.shutdown(); asset_server.server_close(); asset_thread.join(timeout=5)
asset_server = None
asset_thread = None

def start_asset_server():
    global asset_server, asset_thread
    if asset_server is None:
        asset_server = ThreadingHTTPServer(('127.0.0.1', 0), LocalGameHandler)
        asset_thread = threading.Thread(target=asset_server.serve_forever, daemon=True)
        asset_thread.start()
    return f'http://127.0.0.1:{asset_server.server_port}/'

print('Game folder:', GAME_DIR)


## 1. Configuration and access to the local game

The first code cell is the setup cell. It imports the Python libraries, selects CPU inference and training, fixes the Python/NumPy/PyTorch random seeds, and sets the game parameters used by later cells. `ACTION_REPEAT = 4` means one chosen action is applied while the game advances four 60 Hz frames. `GAMMA = 0.99` is the per-frame discount factor used when accumulating rewards.

`GAME_DIR` points to the neighboring `t-rex-runner-gh-pages` folder. The cell checks that the game page and its `state-control.js` interface exist, then defines a small static HTTP server bound to `127.0.0.1` on an automatically chosen port. Firefox can load the local JavaScript and image assets through that server. The server serves the existing files; it does not copy or unpack them. `start_asset_server()` starts it when the game environment is created. The last section shuts it down.

The same setup cell finds `tools/geckodriver.exe` or a geckodriver on `PATH`, and configures the Firefox executable if a non-default installation is needed. If this cell fails, fix the reported path or dependency before running any later cell. If names such as `start_asset_server` are missing, restart the kernel and run cells in order.

## 2. Neural network, frame history, and replay memory

### Q-network

`GitHubDinoNet` is the PyTorch version of the CNN in the Paperspace tutorial. Its input tensor has shape **4 × 80 × 80**: four grayscale game frames stacked as channels. Three convolution and max-pooling stages extract visual features. The result is flattened, passed through a 512-unit dense layer, and mapped to two output values: estimated action values for **0 = continue running** and **1 = jump**. At decision time, the policy selects the action with the larger value.

The helper `_keras_same` reproduces Keras `padding="same"` padding so the convolution geometry matches the original model. The channel reordering before flattening preserves the feature order expected by weights created by the Keras model. These details matter when loading weights trained with that architecture.

### Four-frame state

`stack_frames` stores the newest frame first and the three preceding frames after it. A single still image cannot show whether an obstacle is approaching; the short frame history gives the CNN motion cues. Frames are converted to grayscale, cropped to the game area, resized to 80 × 80, and kept as byte-valued pixels, matching the model’s training convention.

### Prioritized replay

`PrioritizedReplay` keeps recent transitions `(state, action, reward, next_state, terminal, discount)` in a bounded circular buffer. It samples transitions with probability proportional to their priority raised to `alpha`; after learning, each sampled priority is updated from its absolute temporal-difference error. The importance weights returned by sampling compensate for the biased sampling distribution in the loss. The training cell limits the buffer to 2,000 transitions to control RAM use. Replay lets the learner revisit varied past situations instead of learning only from the most recent frames.

In [ ]:
class GitHubDinoNet(nn.Module):
    """PyTorch equivalent of buildmodel() in the GitHub RL notebook."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 32, 8, stride=4, padding=0)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=0)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=0)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64, 512)
        self.fc2 = nn.Linear(512, 2)  # original GitHub actions: 0=run, 1=jump

    @staticmethod
    def _keras_same(conv, x):
        h, w = x.shape[-2:]
        kh, kw = conv.kernel_size
        sh, sw = conv.stride
        oh, ow = (h + sh - 1) // sh, (w + sw - 1) // sw
        ph, pw = max((oh - 1) * sh + kh - h, 0), max((ow - 1) * sw + kw - w, 0)
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2))
        return conv(x)

    def forward(self, x):
        x = F.relu(self.pool1(self._keras_same(self.conv1, x)))
        x = F.relu(self.pool2(self._keras_same(self.conv2, x)))
        x = F.relu(self.pool3(self._keras_same(self.conv3, x)))
        # Restore Keras NHWC flatten order before applying imported Dense weights.
        x = x.permute(0, 2, 3, 1).contiguous().flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

class PrioritizedReplay:
    def __init__(self, cap=50000, alpha=0.6):
        self.items = []
        self.priorities = []
        self.cap = cap
        self.pos = 0
        self.alpha = alpha

    def add(self, item):
        p = max(self.priorities, default=1.0)
        if len(self.items) < self.cap:
            self.items.append(item); self.priorities.append(p)
        else:
            self.items[self.pos] = item; self.priorities[self.pos] = p
            self.pos = (self.pos + 1) % self.cap

    def sample(self, n, beta):
        p = np.asarray(self.priorities, dtype=np.float64) ** self.alpha
        p /= p.sum()
        ids = np.random.choice(len(self.items), n, p=p)
        w = (len(self.items) * p[ids]) ** (-beta); w /= w.max()
        batch = [self.items[i] for i in ids]
        return batch, ids, w.astype(np.float32)

    def update(self, ids, errors):
        for i, e in zip(ids, errors): self.priorities[int(i)] = float(abs(e) + 1e-5)

def stack_frames(frames):
    # The GitHub notebook prepends each new image and shifts older channels right.
    return np.stack(list(frames)[::-1], axis=0)


## 3. Browser game as a reinforcement-learning environment

`LocalGitHubGame` connects Python to the original JavaScript game through Selenium and Firefox. Its `reset(seed)` method reloads the page, waits for the game-control interface, seeds the page’s random-number generator, disables acceleration, and starts a new run. Seeding makes obstacle generation repeatable for a given initialization, although browser timing and software versions can still affect exact reproducibility.

### Controlled game clock

The wrapper replaces the automatic animation-frame loop with `window.__dinoAdvance(n)`. A call advances the original game update function by at most `n` virtual frames at 60 frames per second and returns the current state, rendered canvas, and number of frames actually advanced. This keeps Python’s action and the game transition synchronized: the game does not race ahead while the model is deciding. `ACTION_REPEAT = 4` advances up to four frames per action decision. The wrapper also completes the game’s short intro animation on its virtual clock.

### Observation, action, and reward

The browser canvas is captured as PNG data, decoded in Python, converted to grayscale, cropped, and resized to **80 × 80**. Four such frames form the neural-network input. The policy has two actions: keep running, or issue a jump if the dinosaur is not already airborne. The wrapper returns the next frame, reward, collision flag, score, discounted continuation factor, and number of frames advanced. Each safe frame earns `+0.1`; a collision earns `−1` on the final advanced frame. The training loop adds an extra penalty when a collision occurs below `STABILITY_SCORE`.

The game’s numeric state interface is used for bookkeeping and termination; the CNN still acts on screenshots. During reset, the displayed high-score meter is set to 363 to match the screen appearance expected by the supplied policy. This affects the visual observation supplied to the CNN; it does not alter obstacle physics. Acceleration is held at zero, so experiments concern the fixed-speed version of the game.

In [ ]:
class LocalGitHubGame:
    def __init__(self, seed, action_repeat=ACTION_REPEAT, gamma=GAMMA):
        if GECKODRIVER and not Path(GECKODRIVER).is_file():
            raise FileNotFoundError(f'GECKODRIVER does not exist: {GECKODRIVER}')
        self.action_repeat = max(1, int(action_repeat))
        self.gamma = float(gamma)
        if not callable(globals().get('start_asset_server')):
            raise RuntimeError('Run the first setup cell before creating a game (or use Restart Kernel and Run All).')
        self.url = start_asset_server()
        options = Options()
        if HEADLESS: options.add_argument('-headless')
        if FIREFOX_BINARY: options.binary_location = FIREFOX_BINARY
        options.set_preference('media.volume_scale', '0.0')
        service = (Service(executable_path=GECKODRIVER, log_output=__import__('subprocess').DEVNULL)
                   if GECKODRIVER else Service(log_output=__import__('subprocess').DEVNULL))
        self.driver = webdriver.Firefox(service=service, options=options)
        self.driver.set_window_size(1000, 700)

    def reset(self, seed):
        self.driver.get(self.url)
        deadline = time.monotonic() + 15
        while not self.driver.execute_script('return !!(window.dinoControl && window.dinoControl.getState().ready)'):
            if time.monotonic() > deadline: raise RuntimeError('Game initialization timed out')
            time.sleep(.05)
        # Inject into the page realm so the original obstacle generator sees seeded Math.random.
        self.driver.execute_script("""
          const el=document.createElement('script');
          el.textContent='(()=>{let s='+String(arguments[0]>>>0)+'; Math.random=()=>{s=(1664525*s+1013904223)>>>0;return s/4294967296;};})();';
          document.head.appendChild(el);el.remove();
        """, int(seed))
        self.driver.execute_script("""
          const r=Runner.instance_;r.stop();Runner.config.ACCELERATION=0;r.config.ACCELERATION=0;
          let t=performance.now();
          Object.defineProperty(performance,'now',{configurable:true,value:()=>t});
          window.requestAnimationFrame=()=>1;window.cancelAnimationFrame=()=>{};
          const panel=document.querySelector('body > section');if(panel)panel.style.display='none';
          const message=document.getElementById('messageBox');if(message)message.style.display='none';
          let introStart=null;
          window.__dinoAdvance=(n)=>{
            let advanced=0;
            for(let i=0;i<n && r.playing && !r.crashed;i++){
              if(r.playingIntro && introStart===null)introStart=t;
              if(r.playingIntro && t-introStart>=400){r.startGame();introStart=null;}
              t+=1000/60;r.update();advanced++;
              if(!r.playingIntro)introStart=null;
            }
            return {state:window.dinoControl.getState(),advanced,
              frame:document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]};
          };
          r.time=t;
          const e={keyCode:38,target:document.body,preventDefault(){}};
          r.onKeyDown(e);r.onKeyUp(e);
        """)
        self._ducking = False
        # Match the post-first-game HI pixels seen by the trained policy.
        # This changes only the rendered score display, not game physics.
        self.driver.execute_script("""const r=Runner.instance_;
          r.distanceMeter.setHighScore(363);
          r.distanceMeter.update(0,Math.ceil(r.distanceRan));""")
        return self.observe()

    @staticmethod
    def _process_frame(encoded):
        image = Image.open(io.BytesIO(base64.b64decode(encoded))).convert('L')
        image = image.crop((0, 0, min(500, image.width), min(300, image.height)))
        return np.asarray(image.resize((80, 80), Image.Resampling.BILINEAR), dtype=np.uint8)

    def capture(self):
        encoded = self.driver.execute_script("return document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]")
        return Image.open(io.BytesIO(base64.b64decode(encoded))).convert('RGB')

    def observe(self):
        encoded = self.driver.execute_script("return document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]")
        return self._process_frame(encoded)

    def state(self):
        return self.driver.execute_script('return window.dinoControl.getState()')

    def step(self, action):
        if action not in (0, 1): raise ValueError('Actions: 0=run, 1=jump')
        if action == 1:
            self.driver.execute_script("""const r=Runner.instance_;if(!r.tRex.jumping){
              const e={keyCode:38,target:document.body,preventDefault(){}};r.onKeyDown(e);r.onKeyUp(e);}
            """)
        result = self.driver.execute_script('return window.__dinoAdvance(arguments[0])', self.action_repeat)
        done = bool(result['state']['crashed']); advanced = int(result['advanced'])
        if not done and advanced == 0: raise RuntimeError('Game stopped advancing')
        reward = sum(self.gamma**i * (-1.0 if done and i == advanced-1 else .1) for i in range(advanced))
        return (self._process_frame(result['frame']), reward, done,
                float(result['state']['score']), self.gamma**advanced, advanced)

    def close(self):
        self.driver.quit()


## 4. Optional in-memory training

Set `RUN_TRAINING = True` in the first setup cell before running this section to train a fresh random policy. It uses one browser environment for up to 120 seconds, at most 100 episodes, or 20,000 frames per episode. Set it to `False` to skip training and go directly to the supplied model checkpoint. No model is written to disk in either mode.

The cell initializes a learner, a slowly updated target network, an Adam optimizer, a prioritized replay buffer, and `training_history`. At each episode start, a frozen copy of the learner acts as the behavior policy; weights stay fixed for that episode while the learner trains from collected transitions. Epsilon-greedy actions balance choosing the highest-valued action with occasional random exploration.

Each transition contains four recent grayscale frames, the selected action, reward, next frame stack, terminal flag, and discounted continuation factor. Safe frames receive `+0.1`; a collision receives `-1`, with an additional penalty for collisions below `STABILITY_SCORE`. Once replay has enough examples, Double DQN selects the next action with the learner and estimates its value with the target network. A weighted smooth-L1 loss updates the learner, priorities are refreshed from temporal-difference errors, and the target network is softly updated with `tau = 0.001`.

Episode scores, seeds, frame counts, epsilon, loss, and termination reasons stay in `training_history`. The best completed frozen actor stays in `best_training_actor`. After training, Section 5 still loads `dino_model.pt` into `net`; evaluation and GIF cells therefore use the supplied checkpoint unless you explicitly load the in-memory learner into `net`.

In [ ]:
TRAIN_SECONDS = 120
TRAIN_EPISODES = 100
MAX_GAME_FRAMES = 20000
REPLAY_CAPACITY = 2000
BATCH_SIZE = 64
EARLY_CRASH_PENALTY = 5.0
STABILITY_SCORE = 1000

learner = GitHubDinoNet().to(DEVICE)
target = copy.deepcopy(learner).eval().requires_grad_(False)
optimizer = torch.optim.Adam(learner.parameters(), lr=3e-5)
replay = PrioritizedReplay(cap=REPLAY_CAPACITY)
training_history = []
best_training_actor = None
updates = 0

def train_in_memory():
    global updates, best_training_actor, game
    game = LocalGitHubGame(SEED)
    started = time.monotonic(); total_frames = 0
    for episode in range(1, TRAIN_EPISODES+1):
        if time.monotonic()-started >= TRAIN_SECONDS: break
        actor = copy.deepcopy(learner).eval().requires_grad_(False)
        epsilon = max(.02, .10-.08*total_frames/1_000_000)
        seed = SEED + episode
        observation = game.reset(seed)
        frames = deque([observation.copy() for _ in range(4)], maxlen=4)
        state = stack_frames(frames); losses = []; game_frames = 0; score = 0.; done = False
        while not done and game_frames < MAX_GAME_FRAMES and time.monotonic()-started < TRAIN_SECONDS:
            with torch.inference_mode():
                greedy = int(actor(torch.from_numpy(state).float().unsqueeze(0)).argmax(1).item())
            action = random.randrange(2) if random.random()<epsilon else greedy
            image, reward, done, score, discount, advanced = game.step(action)
            frames.append(image); nxt = stack_frames(frames)
            if done and score < STABILITY_SCORE:
                reward -= EARLY_CRASH_PENALTY * GAMMA**max(0, advanced-1)
            replay.add((state.copy(), action, reward, nxt.copy(), done, discount))
            state = nxt; game_frames += advanced; total_frames += advanced
            if len(replay.items) >= max(100, BATCH_SIZE):
                batch, ids, weights = replay.sample(BATCH_SIZE, min(1., .4+.6*updates/100000))
                st = torch.from_numpy(np.stack([z[0] for z in batch])).float()
                ns = torch.from_numpy(np.stack([z[3] for z in batch])).float()
                actions = torch.tensor([z[1] for z in batch], dtype=torch.long)
                rewards = torch.tensor([z[2] for z in batch], dtype=torch.float32)
                terminals = torch.tensor([z[4] for z in batch], dtype=torch.float32)
                discounts = torch.tensor([z[5] for z in batch], dtype=torch.float32)
                q = learner(st).gather(1, actions[:,None]).squeeze(1)
                with torch.no_grad():
                    next_actions = learner(ns).argmax(1)
                    expected = rewards + discounts*(1-terminals)*target(ns).gather(1,next_actions[:,None]).squeeze(1)
                errors = (q-expected).detach().abs().numpy()
                loss = (torch.from_numpy(weights)*F.smooth_l1_loss(q,expected,reduction='none')).mean()
                if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss')
                optimizer.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(learner.parameters(),10); optimizer.step()
                with torch.no_grad():
                    for tp, lp in zip(target.parameters(), learner.parameters()): tp.lerp_(lp, .001)
                replay.update(ids, errors); updates += 1; losses.append(float(loss.detach()))
        terminal = 'collision' if done else ('frame_limit' if game_frames>=MAX_GAME_FRAMES else 'time_budget')
        row = dict(episode=episode,seed=seed,score=score,frames=game_frames,
                   epsilon=epsilon,mean_loss=float(np.mean(losses)) if losses else None,terminal=terminal)
        training_history.append(row); print(row)
        # Retain only completed episodes; a time-budget-truncated run is not a high-score candidate.
        if terminal != 'time_budget' and (best_training_actor is None or score>best_training_actor['score']):
            best_training_actor = dict(score=score, model=copy.deepcopy(actor.state_dict()), episode=episode)
    game.close()
    return training_history

if RUN_TRAINING:
    train_in_memory()
else:
    print('Training skipped; using the supplied checkpoint later.')


## 5. Load the supplied model weights

This cell creates a network with the same architecture as the training model and reads `dino_model.pt` from the project folder. `torch.load(..., map_location=DEVICE, weights_only=True)` loads the saved tensor data onto the CPU. `load_state_dict` copies those parameters into the CNN; `eval()` disables training-specific behavior and `requires_grad_(False)` prevents gradient tracking during inference.

The cell also checks that the checkpoint’s `action_repeat` matches the environment setting. The model has two actions, so changing the action repeat or the input preprocessing can change its behavior. The checkpoint is read only. This cell sets `net`, which is the model used by the evaluation and animation cells below. It replaces `net` with the supplied model regardless of the learner state created by the training cell.

In [ ]:
net = GitHubDinoNet().to(DEVICE)
checkpoint_bytes = CHECKPOINT.read_bytes()
saved = torch.load(io.BytesIO(checkpoint_bytes), map_location=DEVICE, weights_only=True)
if saved.get('action_repeat', ACTION_REPEAT) != ACTION_REPEAT: raise ValueError('Action repeat mismatch')
net.load_state_dict(saved['model'])
policy_source = f'existing checkpoint: {CHECKPOINT.name}, episode {saved.get("episodes", "unknown")}'
del checkpoint_bytes
net.eval().requires_grad_(False)
print(policy_source)


## 6. Greedy evaluation over independent episodes

`rollout` is the shared episode runner used for evaluation and recording. It resets the game, initializes the four-frame stack, repeatedly selects `argmax(net(state))`, advances the game, and stops on collision or a configured frame/time limit. There is no random exploration and no weight update during rollout, so every action comes from the same frozen `net`.

The evaluation cell runs five episodes with distinct seeds. For each it reports seed, score, frame count, decisions, and termination reason. It then reports the mean score, minimum score, full score list, and number reaching 1,000 points. Showing all scores alongside the mean makes early failures visible. The 20,000-frame and 60-second limits keep evaluation bounded; a time-limited result is not the same as a collision. These measurements apply to the model loaded in Section 5 (the supplied checkpoint by default), not to training episodes and not to the retained best actor.

In [ ]:
EVALUATION_EPISODES = 5
EVALUATION_MAX_FRAMES = 20000
EVALUATION_SECONDS_PER_EPISODE = 60
evaluation_results = []

def rollout(game, seed, max_frames, seconds, record=False, record_limit=600):
    first = game.reset(seed)
    frames = deque([first.copy() for _ in range(4)], maxlen=4)
    images = [game.capture()] if record else []
    image_times = [0.] if record else []
    advanced_total = 0; decisions = 0; score = 0.; done = False
    started = time.monotonic(); next_capture = 0
    # Sample video at <=10 fps; stop recording when bounded capacity is reached.
    while not done and advanced_total < max_frames and time.monotonic()-started < seconds:
        x = torch.from_numpy(stack_frames(frames)).float().unsqueeze(0)
        with torch.inference_mode(): action = int(net(x).argmax(1).item())
        image, reward, done, score, discount, advanced = game.step(action)
        frames.append(image); advanced_total += advanced; decisions += 1
        if record and len(images)<record_limit and advanced_total>=next_capture:
            images.append(game.capture()); image_times.append(advanced_total/60)
            next_capture = advanced_total+24
    if record and round(image_times[-1]*60)<advanced_total:
        images.append(game.capture()); image_times.append(advanced_total/60)
    result = dict(seed=seed,score=score,game_frames=advanced_total,decisions=decisions,
                  terminal='collision' if done else ('frame_limit' if advanced_total>=max_frames else 'time_limit'),
                  policy=policy_source,epsilon=0,learning_updates=0)
    if record:
        result.update(recorded_until_frame=round(image_times[-1]*60),
                      recording_truncated=round(image_times[-1]*60)<advanced_total)
    return result, images, image_times

game = LocalGitHubGame(SEED)
for i in range(EVALUATION_EPISODES):
    row, _, _ = rollout(game, SEED+10000+i, EVALUATION_MAX_FRAMES, EVALUATION_SECONDS_PER_EPISODE)
    evaluation_results.append(row); print(row)
game.close()
scores = [row['score'] for row in evaluation_results]
print(dict(mean=float(np.mean(scores)),minimum=min(scores),scores=scores,
           episodes_above_1000=sum(s>=1000 for s in scores)))


## 7. Record and display one gameplay episode

This cell runs one additional greedy episode with a distinct seed and records screenshots from the game canvas. It samples frames roughly every 24 virtual game frames (about 2.5 images per second at 60 Hz), with a maximum of 900 sampled frames, and appends the final frame when needed. The score and termination details are printed with the result.

The images are combined into an animated GIF using `io.BytesIO`. The GIF bytes are placed in `demo_gif_bytes` and passed to the notebook display as a data URL, so no GIF file is created. The animation shows this single rollout; it is an illustration of the chosen policy, not a substitute for the multi-episode evaluation distribution. The browser is closed after recording.

In [ ]:
demo_result = None
demo_gif_bytes = None
game = LocalGitHubGame(SEED)
demo_result, video_frames, video_times = rollout(game, SEED+20000, 20000, 120, record=True, record_limit=900)
durations = [max(10, round((b-a)*100)*10) for a,b in zip(video_times,video_times[1:])] + [1000]
with io.BytesIO() as buffer:
    video_frames[0].save(buffer, format='GIF', save_all=True, append_images=video_frames[1:],
                         duration=durations,loop=0,optimize=False)
    demo_gif_bytes = buffer.getvalue()
print(demo_result)
display(HTML('<img alt="Actual model-controlled Dino rollout" src="data:image/gif;base64,'+
             base64.b64encode(demo_gif_bytes).decode('ascii')+'">'))
del video_frames, video_times
game.close()


## 8. Stop the local server and release the run

Run this final cell after training, evaluation, and recording. It shuts down the local HTTP server that serves the neighboring game folder and releases its port. Each normal training, evaluation, or demo run closes its Firefox driver when finished. If execution is interrupted before the cell reaches its normal cleanup, close the active `game` object and then run this server cleanup cell.

The model, replay buffer, episode records, evaluation results, and GIF bytes are Python objects in the current kernel. They disappear when the kernel is restarted. The notebook does not save model checkpoints, training logs, or animation files.

In [ ]:
if asset_server is not None:
    asset_server.shutdown(); asset_server.server_close(); asset_thread.join(timeout=5)
    asset_server = None; asset_thread = None
print('Asset server stopped. No application-generated files were saved.')
